In [ ]:
#Limpeza da tabela campeonato-brasileiro-full.csv

# Esse script segue uma ordem lógica de limpeza de dados:
#1. Carregar
#2. Tratar valores nulos "disfarçados"
#3. Limpar texto (espaços, maiúsculas/minúsculas)
#4. Corrigir coluna vencedor
#5. Corrigir datas
#6. Corrigir números da coluna arrecadacao e coluna visitante, mandante.
#7. Trata outliers valores absurdos
#8. Remover duplicatas
#9. Salvar




In [ ]:
import pandas as pd
import numpy as np

#carregamento de dados brutos e bagunçados
df = pd.read_csv(r"D:\projetos git\analise-de-times-br\data\raw\campeonato-brasileiro-full.csv")

#tratamento de valores nulos
fake_null_values = ["", " ", "N/A", "-", "NULL", "null", "NaN", "nan", "<NA>"]
df = df.replace(fake_null_values, np.nan)


In [ ]:
#limpar espaços em branco das colunas
colunas_texto = df.select_dtypes(include="object").columns
for coluna in colunas_texto:
    df[coluna] = df[coluna].str.strip()

#padronizar maisculas e minusculas
coluna_de_sigla = ["mandante_Estado", "visitante_Estado"]
coluna_de_nomes = ["mandante", "visitante", "arena", "tecnico_mandante", "tecnico_visitante"]    
for coluna in coluna_de_nomes:
    df[coluna] = df[coluna].str.title()

for coluna in coluna_de_sigla:
    df[coluna] = df[coluna].str.upper() 

In [ ]:
#correção da  coluna vencedor (todas as colunas que nao tiver vencedor sera colocado empate)
df["vencedor"] = df["vencedor"].fillna("Empate")
df["vencedor"] = df["vencedor"].str.title()

In [ ]:
#correção da coluna de data para formato datetime
#COMENTARIO: A coluna de data possuia diversos fromatos de horario, fazendo com uqe o date time não conseguisse converter todos os valores. A solução desse problema foi criar uma lista de possiveis formatos e converter "manuealmente", até qque todos pudessem virar datetime.
formatos_de_data =  ["%d/%m/%Y", "%Y-%m-%d", "%d.%m.%Y", "%d-%m-%Y","%m/%d/%Y", "%d/%m/%y", "%Y/%m/%d"]
data_convertida = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")

for formato in formatos_de_data:
    ainda_sem_data = data_convertida.isna()
    segunda_tentativa = pd.to_datetime(df.loc[ainda_sem_data, "data"], format = formato, errors="coerce")
    data_convertida.loc[ainda_sem_data] = segunda_tentativa
df["data"] = data_convertida
datas_invalidas = df["data"].isna().sum()

In [ ]:

#correção as colunas de placar transfromando para numeros inteiros e removendo os possiveis textos na coluna mandante e visitante
for coluna in ["mandante_Placar", "visitante_Placar"]:
    df[coluna] = df[coluna].str.replace(" un","", regex=False)
    df[coluna] = df[coluna].str.replace(",","", regex=False)
    df[coluna] = pd.to_numeric(df[coluna], errors="coerce")
    df[coluna] = df[coluna].astype("Int64")


In [ ]:
#correção da coluna de arrecadação (estava em padrão americano e com separador de milhar e foi para o padrão brasileiro)
df["arrecadacao"] = pd.to_numeric(df["arrecadacao"], errors="coerce")

def formatar_arrecadacao(valor):
    if pd.isna(valor):
        return np.nan
    
    texto = f"{valor:,.2f}"
    texto = texto.replace(",", "X")
    texto = texto.replace(".", ",")
    texto = texto.replace("X", ".")
    return f"R$ {texto}"
df["arrecadacao"] = df["arrecadacao"].apply(formatar_arrecadacao)

In [ ]:
#trata outliers valores absurdos
limite_maximo_de_gols = 12
for coluna in ["mandante_Placar", "visitante_Placar"]:
    outliers = df[coluna] > limite_maximo_de_gols
    df.loc[outliers, coluna] = np.nan 

duplicatas = df.duplicated(subset="ID").sum()
df = df.drop_duplicates(subset = "ID", keep = "first")  

In [ ]:
# DECISÃO DE LIMPEZA: as linhas que ficaram com <NA> foram mantidas, pois nao há informação para saber se houve gol ou não. 
# Preencher com 0 poderia gerar uma informação falsa e remover a linha traria danos a interepretação dos dados. 
# Os dados serão importantes para analises futuras.


#ordenação da tabela de acordo com a ordem das colunas e ordenação por data
ordem_das_colunas = [
    "ID", "rodata", "data", "hora","mandante", "visitante","mandante_Placar", "visitante_Placar", "vencedor","mandante_Estado", "visitante_Estado", "arena","formacao_mandante", "formacao_visitante", "tecnico_mandante", "tecnico_visitante", "arrecadacao",]
df = df[ordem_das_colunas]
df = df.sort_values(by=["data"], ascending=True).reset_index(drop=True)



print(f"\nLinhas finais: {len(df)}")
print("\nValores nulos restantes por coluna:")
print(df.isna().sum())


In [ ]:
#salvamento do arqqquivo levando para a pasta processed, que contem os dados limpos.
print(f"\nLinhas finais: {len(df)}")
print("\nValores nulos restantes por coluna:")
print(df.isna().sum())
print("\nDistribuição de tipo_de_gol:")
print(df["tipo_de_gol"].value_counts())


caminho_saida = r"D:\projetos git\analise-de-times-br\data\processed\campeonato-brasileiro-limpo.csv"
df.to_csv(caminho_saida, index=False)

display(df)